# Descriptive Statistics
The goal of this part is to create descriptive statistics and figures for Results

Outputs : 
- outputs/descriptive_stats.txt
- outputs/descriptive_stats.csv
- figures/fig_ai_score_dist.png
- figures/fig_helpful_vote_dist.png
- figures/fig_rating_dist.png
- figures/fig_correlation_matrix.png
- figures/fig_ai_score_by_rating.png

In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from statsmodels.discrete.discrete_model import NegativeBinomialP

# --- Configuration ---
PROJECT_ROOT = Path.home() / "Desktop" / "Dissertation_Analysis"
DATA_DIR = PROJECT_ROOT / "data"
FIG_DIR = PROJECT_ROOT / "figures"
OUT_DIR = PROJECT_ROOT / "outputs"
FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

# Apply academic plotting style
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"

In [66]:
# --- 1. Load data ---
print("Loading scored dataset")
df = pd.read_parquet(DATA_DIR / "reviews_with_ai_scores.parquet")
print(f"  Loaded: {len(df):,} reviews")

# Retain the 4 core variables
key_vars = ["helpful_vote", "AI_generated_score", "rating", "text_length"]
df_analysis = df[key_vars].copy()
print(f"\n  Missing values per variable:")
print(df_analysis.isna().sum())

Loading scored dataset
  Loaded: 700,808 reviews

  Missing values per variable:
helpful_vote          0
AI_generated_score    0
rating                0
text_length           0
dtype: int64


In [67]:
# --- 2. Summary table ---
# Build a descriptive statistics table (mean, SD, min, max, etc.).
# Save the table as a CSV file and a readable text version.
# Quick overdispersion check: compare the variance and the mean of helpful_vote.
print("\nStep 2: Building descriptive statistics table")
desc = df_analysis.describe().T
desc["missing"] = df_analysis.isna().sum().values
desc = desc[["count", "mean", "std", "min", "25%", "50%", "75%", "max", "missing"]]
desc.columns = ["N", "Mean", "SD", "Min", "Q1", "Median", "Q3", "Max", "Missing"]

print(desc.round(3))
desc.to_csv(OUT_DIR / "descriptive_stats.csv")

# Version texte lisible pour la dissertation
with open(OUT_DIR / "descriptive_stats.txt", "w") as f:
    f.write("DESCRIPTIVE STATISTICS\n")
    f.write("=" * 70 + "\n\n")
    f.write(desc.round(3).to_string())
    f.write("\n\n")
    f.write("Rating distribution:\n")
    f.write(df["rating"].value_counts().sort_index().to_string())
    f.write("\n\n")

    # Test de surdispersion pour helpful_vote
    mean_hv = df_analysis["helpful_vote"].mean()
    var_hv = df_analysis["helpful_vote"].var()
    f.write(f"helpful_vote overdispersion check:\n")
    f.write(f"  Mean = {mean_hv:.3f}\n")
    f.write(f"  Variance = {var_hv:.3f}\n")
    f.write(f"  Variance/Mean ratio = {var_hv/mean_hv:.2f}\n")
    f.write(f"  --> Strong overdispersion: Negative Binomial recommended\n"
            if var_hv/mean_hv > 2
            else f"  --> Poisson may be acceptable\n")


Step 2: Building descriptive statistics table
                           N    Mean      SD  Min   Q1  Median      Q3  \
helpful_vote        700808.0   0.924   5.474  0.0  0.0   0.000   1.000   
AI_generated_score  700808.0   0.113   0.274  0.0  0.0   0.001   0.032   
rating              700808.0   3.960   1.494  1.0  3.0   5.000   5.000   
text_length         700808.0  32.784  45.985  1.0  8.0  19.000  40.000   

                         Max  Missing  
helpful_vote         646.000        0  
AI_generated_score     0.999        0  
rating                 5.000        0  
text_length         2585.000        0  


In [68]:
# --- 3. 'AI_generated_score' distribution ---
# Plot the distribution of the AI score across all reviews.
# The red dashed line marks the 0.5 decision threshold.
print("\nStep 3: Plotting AI_generated_score distribution")
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df["AI_generated_score"].dropna(), bins=50,
        color="#1f77b4", edgecolor="black", alpha=0.8)
ax.set_xlabel("AI generation probability (RoBERTa output)")
ax.set_ylabel("Number of reviews")
ax.set_title("Distribution of AI_generated_score across reviews")
ax.axvline(0.5, color="red", linestyle="--", linewidth=1,
           label="Decision threshold (0.5)")
ax.legend()
plt.savefig(FIG_DIR / "fig_ai_score_dist.png")
plt.close()


Step 3: Plotting AI_generated_score distribution


In [69]:
# --- 4. 'helpful_vote' distribution (log scale to show the zero-inflation) ---
# Plot the distribution of helpful_vote.
# Use a log scale because most reviews have zero votes (zero-inflation).
# The right plot zooms on reviews that received at least one vote.
print("Step 4: Plotting helpful_vote distribution")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["helpful_vote"], bins=50, color="#ff7f0e",
             edgecolor="black", alpha=0.8)
axes[0].set_xlabel("helpful_vote")
axes[0].set_ylabel("Number of reviews")
axes[0].set_title("helpful_vote (linear scale)")
axes[0].set_yscale("log")

# Zoom on positive tail (helpful_vote > 0)
axes[1].hist(df[df["helpful_vote"] > 0]["helpful_vote"], bins=50,
             color="#ff7f0e", edgecolor="black", alpha=0.8)
axes[1].set_xlabel("helpful_vote (> 0 only)")
axes[1].set_ylabel("Number of reviews")
axes[1].set_title("Positive helpful_vote (zoomed)")
axes[1].set_yscale("log")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_helpful_vote_dist.png")
plt.close()

Step 4: Plotting helpful_vote distribution


In [70]:
# --- 5. Rating distribution ---
# Plot how many reviews fall in each star rating (1 to 5).
print("Step 5: Plotting rating distribution")
fig, ax = plt.subplots(figsize=(7, 4.5))
rating_counts = df["rating"].value_counts().sort_index()
ax.bar(rating_counts.index, rating_counts.values,
       color="#2ca02c", edgecolor="black", alpha=0.8)
ax.set_xlabel("Star rating")
ax.set_ylabel("Number of reviews")
ax.set_title("Distribution of ratings (All Beauty category)")
for i, v in enumerate(rating_counts.values):
    ax.text(rating_counts.index[i], v + max(rating_counts.values) * 0.01,
            f"{v:,}", ha="center", fontsize=9)
plt.savefig(FIG_DIR / "fig_rating_dist.png")
plt.close()

Step 5: Plotting rating distribution


In [71]:
# --- 6. Correlation matrices ---
# Compute and plot the correlations between the variables.
# Pearson (linear) on the left, Spearman (rank) on the right.
print("Step 6: Building correlation matrices")
df_corr = df_analysis.dropna()
corr_p = df_corr.corr(method="pearson")
corr_s = df_corr.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(corr_p, annot=True, fmt=".3f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=axes[0], cbar_kws={"shrink": 0.8})
axes[0].set_title("Pearson correlations")
sns.heatmap(corr_s, annot=True, fmt=".3f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=axes[1], cbar_kws={"shrink": 0.8})
axes[1].set_title("Spearman correlations")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_correlation_matrix.png")
plt.close()

Step 6: Building correlation matrices


In [72]:
# --- 7. Mean 'AI_score' by rating class ---
# Compute and plot the correlations between the variables.
# Pearson (linear) on the left, Spearman (rank) on the right.
print("Step 7: AI_score by rating class")
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="rating", y="AI_generated_score",
            ax=ax, showfliers=False, color="#9467bd")
ax.set_xlabel("Star rating")
ax.set_ylabel("AI generation probability")
ax.set_title("AI_generated_score by rating class")
plt.savefig(FIG_DIR / "fig_ai_score_by_rating.png")
plt.close()

# Numerical statistics
print("\n  Mean AI_Score by rating:")
print(df.groupby("rating")["AI_generated_score"].agg(["mean", "median", "std", "count"]))

Step 7: AI_score by rating class

  Mean AI_Score by rating:
            mean    median       std   count
rating                                      
1.0     0.089044  0.000464  0.249690  101948
2.0     0.110547  0.000734  0.276110   43009
3.0     0.113252  0.000870  0.278968   56283
4.0     0.109299  0.001077  0.269591   79318
5.0     0.119301  0.000931  0.279313  420250


In [73]:
# --- 8. Counting "AI-suspect" reviews ---
# Count the reviews that look AI-generated (score above 0.5).
# Add these results to the text file.
threshold = 0.5
n_ai = (df["AI_generated_score"] > threshold).sum()
pct_ai = n_ai / len(df) * 100
print(f"\nReviews with AI_score > {threshold}: {n_ai:,} ({pct_ai:.2f}%)")

with open(OUT_DIR / "descriptive_stats.txt", "a") as f:
    f.write(f"\n\nAI-suspect reviews (score > {threshold}):\n")
    f.write(f"  N = {n_ai:,} ({pct_ai:.2f}% of total)\n")
    f.write("\nMean AI_score by rating class:\n")
    f.write(df.groupby("rating")["AI_generated_score"]
              .agg(["mean", "median", "std", "count"]).round(3).to_string())

print("\n" + "=" * 60)
print("Descriptive analysis complete.")
print(f"  Tables  -> {OUT_DIR}")
print(f"  Figures -> {FIG_DIR}")
print("=" * 60)


Reviews with AI_score > 0.5: 67,364 (9.61%)

Descriptive analysis complete.
  Tables  -> /Users/brg_elise/Desktop/Dissertation_Analysis/outputs
  Figures -> /Users/brg_elise/Desktop/Dissertation_Analysis/figures


# Regression

In [74]:
"""
Regression analysis: count model with interaction terms.

Model tested:
    helpful_vote ~ AI_c * rating_c + AI_c * length_c

Three model families are compared (selection by AIC):
    1. Poisson
    2. Negative Binomial
    3. OLS (robustness check)
 
Outputs :
- outputs/regression_summary.txt : résumés complets des trois modèles
- outputs/regression_coefficients.csv : table machine-readable
- outputs/model_comparison.txt : AIC, BIC, recommandation
- figures/fig_interaction_ai_rating.png
- figures/fig_interaction_ai_length.png
- figures/fig_residual_diagnostics.png
"""
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pathlib import Path
 
PROJECT_ROOT = Path.home() / "Desktop" / "Dissertation_Analysis"
DATA_DIR = PROJECT_ROOT / "data"
FIG_DIR = PROJECT_ROOT / "figures"
OUT_DIR = PROJECT_ROOT / "outputs"

sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"

In [75]:
# 1. Data Preparation
print("Step 1: Loading data")
df = pd.read_parquet(DATA_DIR / "reviews_with_ai_scores.parquet")
print(f"  Loaded: {len(df):,} reviews")
 
cols_model = ["helpful_vote", "AI_generated_score", "rating", "text_length"]
df_model = df[cols_model].dropna().copy()
print(f"  After dropping NaN: {len(df_model):,} reviews")
 
# enter the continuous variables (subtract the mean).
# Centering makes the interaction terms easier to interpret.
df_model["AI_c"] = df_model["AI_generated_score"] - df_model["AI_generated_score"].mean()
df_model["rating_c"] = df_model["rating"] - df_model["rating"].mean()
df_model["length_c"] = df_model["text_length"] - df_model["text_length"].mean()
 
# Also create a log version of length, because length is very skewed.
# Define the model formula with the two interaction terms.
df_model["log_length_c"] = (
    np.log1p(df_model["text_length"]) - np.log1p(df_model["text_length"]).mean()
)
 
formula_main = "helpful_vote ~ AI_c * rating_c + AI_c * log_length_c"

Step 1: Loading data
  Loaded: 700,808 reviews
  After dropping NaN: 700,808 reviews


In [76]:
# 2. VIF — Detection of multicollinearity
# Check multicollinearity with the Variance Inflation Factor (VIF).
# VIF values close to 1 mean the predictors are not redundant.
print("\nStep 2: Variance Inflation Factors (multicollinearity check)")
X_vif = df_model[["AI_c", "rating_c", "log_length_c"]].copy()
X_vif = sm.add_constant(X_vif)
 
vif_df = pd.DataFrame({
    "variable": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i)
            for i in range(X_vif.shape[1])],
})
print(vif_df.round(3).to_string(index=False))


Step 2: Variance Inflation Factors (multicollinearity check)
    variable   VIF
       const 1.000
        AI_c 1.001
    rating_c 1.008
log_length_c 1.007


In [77]:
# 3. Poisson Model
# Fit a Poisson regression (the first count model).
print("\nStep 3: Fitting Poisson GLM")
m_poisson = smf.glm(
    formula_main,
    data=df_model,
    family=sm.families.Poisson(),
).fit()
print(m_poisson.summary())
 
# Overdispersion test: Pearson chi2 divided by the residual degrees of freedom.
# A value far above 1 means Poisson is not enough, so we move to Negative Binomial.
pearson_chi2 = m_poisson.pearson_chi2
df_resid = m_poisson.df_resid
dispersion = pearson_chi2 / df_resid
print(f"\n  Dispersion statistic (Pearson chi2 / df_resid): {dispersion:.3f}")
print(f"  (Values >> 1 indicate overdispersion --> use Negative Binomial)")


Step 3: Fitting Poisson GLM
                 Generalized Linear Model Regression Results                  
Dep. Variable:           helpful_vote   No. Observations:               700808
Model:                            GLM   Df Residuals:                   700802
Model Family:                 Poisson   Df Model:                            5
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.4518e+06
Date:                Sun, 21 Jun 2026   Deviance:                   2.4194e+06
Time:                        18:16:37   Pearson chi2:                 1.53e+07
No. Iterations:                     7   Pseudo R-squ. (CS):             0.4853
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept

In [78]:
# 4. Negative Binomial Model

print("\nStep 4: Fitting Negative Binomial GLM (alpha estimated)")

# Step 4a: the statsmodels GLM cannot estimate alpha; it fixes it to 1.0 by default.
#          So we first estimate alpha by maximum likelihood with the NB2 count model.
m_nb_mle = smf.negativebinomial(formula_main, data=df_model).fit(disp=False)
alpha_hat = m_nb_mle.params["alpha"]
print(f"  Estimated alpha (NB2, MLE): {alpha_hat:.4f}")

# Step 4b: refit the GLM with this estimated alpha.
#          We keep a GLM object to reuse .predict, .resid_pearson, etc. later.
m_nb = smf.glm(
    formula_main,
    data=df_model,
    family=sm.families.NegativeBinomial(alpha=alpha_hat),
).fit()
print(m_nb.summary())


Step 4: Fitting Negative Binomial GLM (alpha estimated)


/Users/brg_elise/Desktop/Dissertation_Analysis/venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: divide by zero encountered in log
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/Users/brg_elise/Desktop/Dissertation_Analysis/venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: invalid value encountered in multiply
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/Users/brg_elise/Desktop/Dissertation_Analysis/venv/lib/python3.14/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


  Estimated alpha (NB2, MLE): 0.0000
                 Generalized Linear Model Regression Results                  
Dep. Variable:           helpful_vote   No. Observations:               700808
Model:                            GLM   Df Residuals:                   700802
Model Family:        NegativeBinomial   Df Model:                            5
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.4518e+06
Date:                Sun, 21 Jun 2026   Deviance:                   2.4194e+06
Time:                        18:16:38   Pearson chi2:                 1.53e+07
No. Iterations:                     8   Pseudo R-squ. (CS):             0.4853
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
I

In [79]:
# 5. OLS comme robustness check
# Fit an OLS model as a robustness check, with HC3 robust standard errors.
print("\nStep 5: Fitting OLS (robustness check)")
m_ols = smf.ols(formula_main, data=df_model).fit(cov_type="HC3")
print(m_ols.summary())


Step 5: Fitting OLS (robustness check)
                            OLS Regression Results                            
Dep. Variable:           helpful_vote   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     1007.
Date:                Sun, 21 Jun 2026   Prob (F-statistic):               0.00
Time:                        18:16:39   Log-Likelihood:            -2.1784e+06
No. Observations:              700808   AIC:                         4.357e+06
Df Residuals:                  700802   BIC:                         4.357e+06
Df Model:                           5                                         
Covariance Type:                  HC3                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------

In [80]:
# 6. Models Comparison
# Compare the three models with AIC, pseudo-R² / R², and log-likelihood.
# The model with the lowest AIC is the best fit.
# Save the comparison table.

print("\nStep 6: Model comparison")
comp = pd.DataFrame({
    "Model": ["Poisson", "Negative Binomial", "OLS (HC3)"],
    "AIC": [m_poisson.aic, m_nb.aic, m_ols.aic],
    "Pseudo_R2 / R2": [
        1 - m_poisson.llf / m_poisson.llnull,
        1 - m_nb.llf / m_nb.llnull,
        m_ols.rsquared,
    ],
    "LogLik": [m_poisson.llf, m_nb.llf, m_ols.llf],
})
print(comp.round(3).to_string(index=False))
comp.to_csv(OUT_DIR / "model_comparison.csv", index=False)


Step 6: Model comparison
            Model         AIC  Pseudo_R2 / R2       LogLik
          Poisson 2903534.149           0.138 -1451761.075
Negative Binomial 2903534.048           0.138 -1451761.024
        OLS (HC3) 4356886.995           0.021 -2178437.497


In [2]:
# 7. Saving Summaries
# Save the full summaries of the three models to a text file.
# Build a clean coefficient table for the main (NB) model.
# Add the IRR (exp of the coefficient) and its confidence interval.
# The IRR shows how each variable multiplies the expected number of votes.
# Save the coefficient table as a CSV file.

print("\nStep 7: Saving regression outputs")
 
with open(OUT_DIR / "regression_summary.txt", "w") as f:
    f.write("REGRESSION ANALYSIS — FULL SUMMARIES\n")
    f.write("=" * 70 + "\n\n")
    f.write("Formula: " + formula_main + "\n\n")
    f.write("Variance Inflation Factors:\n")
    f.write(vif_df.round(3).to_string(index=False))
    f.write("\n\n" + "=" * 70 + "\n\n")
    f.write("MODEL 1: POISSON\n" + "-" * 70 + "\n")
    f.write(str(m_poisson.summary()))
    f.write(f"\n\nDispersion statistic: {dispersion:.3f}\n\n")
    f.write("=" * 70 + "\n\n")
    f.write("MODEL 2: NEGATIVE BINOMIAL\n" + "-" * 70 + "\n")
    f.write(str(m_nb.summary()))
    f.write("\n\n" + "=" * 70 + "\n\n")
    f.write("MODEL 3: OLS with HC3 robust SE (robustness check)\n" + "-" * 70 + "\n")
    f.write(str(m_ols.summary()))
    f.write("\n\n" + "=" * 70 + "\n\n")
    f.write("MODEL COMPARISON\n" + "-" * 70 + "\n")
    f.write(comp.round(3).to_string(index=False))
    f.write("\n\n")
    best_model = comp.loc[comp["AIC"].idxmin(), "Model"]
    f.write(f"BEST MODEL (lowest AIC): {best_model}\n")
 
# Coefficients du modèle principal (Negative Binomial)
params = m_nb_mle.params.drop("alpha")
coef_df = pd.DataFrame({
    "term": params.index,
    "coef": params.values,
    "std_err": m_nb.bse.drop("alpha").values,
    "z": m_nb.tvalues.drop("alpha").values,
    "p_value": m_nb.pvalues.drop("alpha").values,
    "ci_low": m_nb.conf_int().drop("alpha")[0].values,
    "ci_high": m_nb.conf_int().drop("alpha")[1].values,
})
coef_df["IRR"] = np.exp(coef_df["coef"])  # Incidence Rate Ratios
coef_df["IRR_ci_low"] = np.exp(coef_df["ci_low"])
coef_df["IRR_ci_high"] = np.exp(coef_df["ci_high"])
coef_df.to_csv(OUT_DIR / "regression_coefficients.csv", index=False)
print(f"  Saved: regression_summary.txt, regression_coefficients.csv")


Step 7: Saving regression outputs


NameError: name 'OUT_DIR' is not defined

In [ ]:
# 8. Interaction Figures
print("\nStep 8: Generating interaction plots")
 
# 8a. AI score x rating: predict votes for ratings 1, 3 and 5, at mean length.
ai_range = np.linspace(0, 1, 100)
ai_c_range = ai_range - df_model["AI_generated_score"].mean()
 
fig, ax = plt.subplots(figsize=(8, 5))
for r in [1, 3, 5]:
    r_c = r - df_model["rating"].mean()
    pred_df = pd.DataFrame({
        "AI_c": ai_c_range,
        "rating_c": r_c,
        "log_length_c": 0,  
    })
    pred = m_nb.predict(pred_df)
    ax.plot(ai_range, pred, label=f"Rating = {r}", linewidth=2)
ax.set_xlabel("AI generation probability")
ax.set_ylabel("Predicted helpful votes")
ax.set_title("Interaction: AI score × Rating (at mean text length)")
ax.legend(title="Star rating")
plt.savefig(FIG_DIR / "fig_interaction_ai_rating.png")
plt.close()
 
# 8b. AI score x length: predict votes for short, medium and long reviews, at mean rating.
length_percentiles = df_model["text_length"].quantile([0.25, 0.5, 0.75]).values
fig, ax = plt.subplots(figsize=(8, 5))
labels = ["Short (Q1)", "Medium (Median)", "Long (Q3)"]
for length, label in zip(length_percentiles, labels):
    log_length_c = np.log1p(length) - np.log1p(df_model["text_length"]).mean()
    pred_df = pd.DataFrame({
        "AI_c": ai_c_range,
        "rating_c": 0, 
        "log_length_c": log_length_c,
    })
    pred = m_nb.predict(pred_df)
    ax.plot(ai_range, pred,
            label=f"{label} ({int(length)} words)", linewidth=2)
ax.set_xlabel("AI generation probability")
ax.set_ylabel("Predicted helpful votes")
ax.set_title("Interaction: AI score × Review length (at mean rating)")
ax.legend(title="Review length")
plt.savefig(FIG_DIR / "fig_interaction_ai_length.png")
plt.close()
 
# 8c. Residual diagnostics: residuals vs fitted values, and a Q-Q plot.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fitted = m_nb.fittedvalues
resid = m_nb.resid_pearson
axes[0].scatter(fitted, resid, alpha=0.3, s=5)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Pearson residuals")
axes[0].set_title("Residuals vs fitted (Negative Binomial)")
sm.qqplot(resid, line="45", ax=axes[1], markersize=3, alpha=0.3)
axes[1].set_title("Q-Q plot of Pearson residuals")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_residual_diagnostics.png")
plt.close()
 
print("\n" + "=" * 60)
print("Regression analysis complete.")
print(f"  Tables  -> {OUT_DIR}")
print(f"  Figures -> {FIG_DIR}")
print("=" * 60)
print(f"\nRecommended model (lowest AIC): {best_model}")


Step 8: Generating interaction plots

Regression analysis complete.
  Tables  -> /Users/brg_elise/Desktop/Dissertation_Analysis/outputs
  Figures -> /Users/brg_elise/Desktop/Dissertation_Analysis/figures

Recommended model (lowest AIC): Negative Binomial
